# Kernel Perceptron

This notebook follows the lecture derivation from the ordinary Perceptron in feature space to the kernel Perceptron representation.

## 1. Perceptron in feature space

Whenever example $i$ is misclassified, the feature-space update is

$$\theta \leftarrow \theta + y_i\phi(x_i).$$

Starting from $\theta=0$, the final parameter can therefore be written as

$$\theta=\sum_j\alpha_jy_j\phi(x_j),$$

where $\alpha_j$ counts how many times example $j$ caused an update.

In [ ]:
import numpy as np

def phi(X):
    X = np.asarray(X)
    return np.column_stack([X[:, 0], X[:, 1], X[:, 0] * X[:, 1]])

X = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]], dtype=float)
y = np.array([1, -1, -1, 1])
X_phi = phi(X)

def perceptron_feature_space(X_phi, y, epochs=10):
    theta = np.zeros(X_phi.shape[1])
    for _ in range(epochs):
        for xi, yi in zip(X_phi, y):
            if yi * (theta @ xi) <= 0:
                theta += yi * xi
    return theta

theta = perceptron_feature_space(X_phi, y)
print('Feature-space theta:', theta)
print('Training scores:', X_phi @ theta)
print('Predictions:', np.where(X_phi @ theta >= 0, 1, -1))

## 2. Rewrite prediction using inner products

For a new example $x$,

$$\theta^T\phi(x)=\sum_j\alpha_jy_j\phi(x_j)^T\phi(x).$$

Defining $K(x_j,x)=\phi(x_j)^T\phi(x)$ gives

$$\theta^T\phi(x)=\sum_j\alpha_jy_jK(x_j,x).$$

In [ ]:
def feature_map_kernel_matrix(A, B):
    A_phi = phi(A)
    B_phi = phi(B)
    return A_phi @ B_phi.T

def kernel_perceptron(X, y, epochs=10):
    n = len(X)
    alpha = np.zeros(n, dtype=int)
    K = feature_map_kernel_matrix(X, X)

    for _ in range(epochs):
        for i in range(n):
            score = np.sum(alpha * y * K[:, i])
            if y[i] * score <= 0:
                alpha[i] += 1
    return alpha, K

alpha, K = kernel_perceptron(X, y)
kernel_scores = K.T @ (alpha * y)

print('Alpha:', alpha)
print('Kernel scores:', kernel_scores)
print('Predictions:', np.where(kernel_scores >= 0, 1, -1))

## 3. Compare the two representations

The feature-space and kernel implementations make the same predictions because they use exactly the same inner products, with the kernel computed from the feature map.

In [ ]:
feature_scores = X_phi @ theta

print('Feature-space scores:', feature_scores)
print('Kernel scores:       ', kernel_scores)
print('Scores agree:', np.allclose(feature_scores, kernel_scores))
print('Predictions agree:', np.array_equal(
    np.where(feature_scores >= 0, 1, -1),
    np.where(kernel_scores >= 0, 1, -1)
))
assert np.allclose(feature_scores, kernel_scores)

## 4. The kernel Perceptron update

Initialize all coefficients to zero:

$$\alpha_j=0.$$

For training example $i$, compute

$$s_i=\sum_j\alpha_jy_jK(x_j,x_i).$$

If $y_is_i\le0$, increment

$$\alpha_i\leftarrow\alpha_i+1.$$

The high-dimensional parameter vector is never explicitly constructed.

### Takeaway

The kernel Perceptron is not a different learning principle. It is the ordinary Perceptron rewritten so that feature-space inner products are evaluated by a kernel.